In [7]:
# ============================================================
# GOOGLE PLAY STORE - BUBBLE CHART
# ============================================================

import pandas as pd
import plotly.express as px
from datetime import datetime
from zoneinfo import ZoneInfo

# ------------------------------------------------------------
# 1. LOAD CSV FILE
# ------------------------------------------------------------

file_path = r"C:\Users\Harshitha.M\Downloads\Play Store Data (1).csv"

df = pd.read_csv(file_path)

print("File loaded successfully!")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())


# ------------------------------------------------------------
# 2. CLEAN COLUMN NAMES
# ------------------------------------------------------------

df.columns = df.columns.str.strip()


# ------------------------------------------------------------
# 3. CLEAN RATING
# ------------------------------------------------------------

df["Rating"] = pd.to_numeric(
    df["Rating"],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. CLEAN REVIEWS
# ------------------------------------------------------------

df["Reviews"] = pd.to_numeric(
    df["Reviews"],
    errors="coerce"
)


# ------------------------------------------------------------
# 5. CLEAN INSTALLS
# ------------------------------------------------------------

df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

df["Installs"] = pd.to_numeric(
    df["Installs"],
    errors="coerce"
)


# ------------------------------------------------------------
# 6. CONVERT APP SIZE TO MB
# ------------------------------------------------------------

def size_to_mb(value):

    value = str(value).strip()

    if value.lower() == "varies with device":
        return None

    try:

        if value.lower().endswith("k"):
            return float(value[:-1]) / 1024

        elif value.lower().endswith("m"):
            return float(value[:-1])

        else:
            return float(value)

    except:
        return None


df["Size_MB"] = df["Size"].apply(size_to_mb)


# ------------------------------------------------------------
# 7. FILTER CATEGORIES
# ------------------------------------------------------------

categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL"
]

df["Category_Clean"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# 8. APPLY REQUIRED FILTERS
# ------------------------------------------------------------

filtered = df[
    (df["Rating"] > 3.5)
    &
    (df["Reviews"] > 500)
    &
    (df["Installs"] > 50000)
    &
    (df["Category_Clean"].isin(categories))
    &
    (
        ~df["App"]
        .astype(str)
        .str.contains("S", case=False, na=False)
    )
    &
    (df["Size_MB"].notna())
].copy()


# ------------------------------------------------------------
# 9. CHECK SENTIMENT SUBJECTIVITY
# ------------------------------------------------------------

subjectivity_column = None

for column in df.columns:

    if "subjectivity" in column.lower():
        subjectivity_column = column
        break


if subjectivity_column is not None:

    filtered[subjectivity_column] = pd.to_numeric(
        filtered[subjectivity_column],
        errors="coerce"
    )

    filtered = filtered[
        filtered[subjectivity_column] > 0.5
    ]

    print("Sentiment Subjectivity > 0.5 applied.")

else:

    print(
        "Note: Sentiment Subjectivity column is not present "
        "in this CSV, so that condition was skipped."
    )


# ------------------------------------------------------------
# 10. TRANSLATE REQUIRED CATEGORIES
# ------------------------------------------------------------

translation = {

    "GAME": "Game",

    "BEAUTY": "सौंदर्य",

    "BUSINESS": "வணிகம்",

    "COMICS": "Comics",

    "COMMUNICATION": "Communication",

    "DATING": "Partnersuche",

    "ENTERTAINMENT": "Entertainment",

    "SOCIAL": "Social"
}


filtered["Category_Display"] = (
    filtered["Category_Clean"]
    .map(translation)
)


# ------------------------------------------------------------
# 11. REMOVE DUPLICATE APPS
# ------------------------------------------------------------

filtered = filtered.drop_duplicates(
    subset=["App"]
)


# ------------------------------------------------------------
# 12. DISPLAY NUMBER OF RESULTS
# ------------------------------------------------------------

print()
print("==========================================")
print("FILTERED APPLICATIONS:", len(filtered))
print("==========================================")


# ------------------------------------------------------------
# 13. CHECK INDIA TIME
# ------------------------------------------------------------

india_time = datetime.now(
    ZoneInfo("Asia/Kolkata")
)

minutes_now = (
    india_time.hour * 60
    + india_time.minute
)

five_pm = 17 * 60
seven_pm = 19 * 60

print(
    "Current IST time:",
    india_time.strftime("%I:%M:%S %p")
)

print("Graph time: 5:00 PM - 7:00 PM IST")


# ------------------------------------------------------------
# 14. SHOW GRAPH ONLY BETWEEN 5 PM AND 7 PM
# ------------------------------------------------------------

if five_pm <= minutes_now < seven_pm:

    if len(filtered) == 0:

        print("No applications match the filters.")

    else:

        # ----------------------------------------------------
        # BUBBLE CHART
        # ----------------------------------------------------

        fig = px.scatter(

            filtered,

            x="Size_MB",

            y="Rating",

            size="Installs",

            color="Category_Display",

            hover_name="App",

            hover_data={
                "Size_MB": ":.2f",
                "Rating": ":.2f",
                "Reviews": ":,.0f",
                "Installs": ":,.0f"
            },

            size_max=55,

            title="Google Play Store Bubble Chart",

            labels={
                "Size_MB": "App Size (MB)",
                "Rating": "Average Rating",
                "Installs": "Number of Installs",
                "Category_Display": "Category"
            }
        )


        # ----------------------------------------------------
        # MAKE GAME PINK
        # ----------------------------------------------------

        for trace in fig.data:

            if trace.name == "Game":

                trace.marker.color = "pink"


        # ----------------------------------------------------
        # GRAPH DESIGN
        # ----------------------------------------------------

        fig.update_layout(

            width=1100,

            height=700,

            template="plotly_white",

            title={
                "text": "Google Play Store Bubble Chart",
                "x": 0.5
            },

            xaxis={
                "title": "App Size (MB)",
                "showgrid": True
            },

            yaxis={
                "title": "Average Rating",
                "range": [3.5, 5.1],
                "showgrid": True
            },

            legend_title="Category"
        )


        # ----------------------------------------------------
        # DISPLAY GRAPH
        # ----------------------------------------------------

        fig.show()


        # ----------------------------------------------------
        # SHOW FILTERED DATA
        # ----------------------------------------------------

        print()
        print("Top filtered applications:")

        display(
            filtered[
                [
                    "App",
                    "Category_Display",
                    "Size_MB",
                    "Rating",
                    "Reviews",
                    "Installs"
                ]
            ]
            .sort_values(
                "Installs",
                ascending=False
            )
            .head(20)
        )

else:

    print()
    print("==========================================")
    print("GRAPH NOT DISPLAYED")
    print("==========================================")
    print(
        "The graph is configured to display only "
        "between 5:00 PM and 7:00 PM IST."
    )

File loaded successfully!
Rows: 10841
Columns: ['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']
Note: Sentiment Subjectivity column is not present in this CSV, so that condition was skipped.

FILTERED APPLICATIONS: 308
Current IST time: 07:08:31 PM
Graph time: 5:00 PM - 7:00 PM IST

GRAPH NOT DISPLAYED
The graph is configured to display only between 5:00 PM and 7:00 PM IST.
